In [5]:
# ============================================
# PHASE 1 — BASELINE MODEL
# STEP 4 — FEATURE ENGINEERING
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Load raw dataset
DATA_PATH = Path("../data/application_train.csv")
df = pd.read_csv(DATA_PATH)

# Separate features and target
X = df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()
y = df["TARGET"].copy()

# Fix the known Home Credit sentinel value
# 365243 represents an abnormal/special employment value.
X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, np.nan)

# Recreate the SAME 80/20 stratified split used in Step 3.
# random_state=42 ensures we get the same split.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create copies for feature engineering
X_train_fe = X_train.copy()
X_valid_fe = X_valid.copy()

print("Training shape:", X_train_fe.shape)
print("Validation shape:", X_valid_fe.shape)
print("Training default rate:", y_train.mean())
print("Validation default rate:", y_valid.mean())

Training shape: (246008, 120)
Validation shape: (61503, 120)
Training default rate: 0.08072908198107379
Validation default rate: 0.08072776937710356


In [6]:
#Create a reusable feature-engineering function cause we can use it later on the validation set and test set.

import numpy as np


def engineer_features(df):
    """
    Create a small set of domain-informed features
    for the baseline credit-risk model.

    No target information is used here.
    """

    data = df.copy()

    # --------------------------------------------
    # 1. Age in years
    # DAYS_BIRTH is stored as negative days.
    # --------------------------------------------
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25

    # --------------------------------------------
    # 2. Employment duration in years
    # DAYS_EMPLOYED was already cleaned in Step 3.
    # --------------------------------------------
    data["EMPLOYED_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25

    # --------------------------------------------
    # Safe division helper
    # Prevents division-by-zero problems.
    # --------------------------------------------
    def safe_divide(a, b):
        return a / b.replace(0, np.nan)

    # --------------------------------------------
    # 3. Credit relative to income
    # How large is the requested credit compared
    # with the applicant's annual income?
    # --------------------------------------------
    data["CREDIT_TO_INCOME"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_INCOME_TOTAL"]
    )

    # --------------------------------------------
    # 4. Annuity relative to income
    # Approximate repayment burden relative to income.
    # --------------------------------------------
    data["ANNUITY_TO_INCOME"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_INCOME_TOTAL"]
    )

    # --------------------------------------------
    # 5. Credit relative to annuity
    # Approximate relationship between loan amount
    # and scheduled payment.
    # --------------------------------------------
    data["CREDIT_TO_ANNUITY"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_ANNUITY"]
    )

    # --------------------------------------------
    # 6. Goods price relative to credit
    # Relationship between requested credit and
    # underlying goods price.
    # --------------------------------------------
    data["GOODS_TO_CREDIT"] = safe_divide(
        data["AMT_GOODS_PRICE"],
        data["AMT_CREDIT"]
    )

    # --------------------------------------------
    # 7. Income per family member
    # Approximate income available per household member.
    # --------------------------------------------
    data["INCOME_PER_FAMILY_MEMBER"] = safe_divide(
        data["AMT_INCOME_TOTAL"],
        data["CNT_FAM_MEMBERS"]
    )

    # --------------------------------------------
    # 8. Income per child
    # Use children + 1 to avoid division by zero.
    # --------------------------------------------
    data["INCOME_PER_CHILD"] = (
        data["AMT_INCOME_TOTAL"] /
        (data["CNT_CHILDREN"] + 1)
    )

    # --------------------------------------------
    # 9. External credit-score average
    # Combine the three external sources when available.
    # --------------------------------------------
    external_sources = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    data["EXT_SOURCE_MEAN"] = data[
        external_sources
    ].mean(axis=1)

    # --------------------------------------------
    # 10. Strongest external score available
    # Useful when one or more external sources are missing.
    # --------------------------------------------
    data["EXT_SOURCE_MAX"] = data[
        external_sources
    ].max(axis=1)

    return data

In [7]:
#apply to train and validate

X_train_fe = engineer_features(X_train_fe)
X_valid_fe = engineer_features(X_valid_fe)

print("Training shape after feature engineering:", X_train_fe.shape)
print("Validation shape after feature engineering:", X_valid_fe.shape)

Training shape after feature engineering: (246008, 130)
Validation shape after feature engineering: (61503, 130)


In [8]:
#New Features to check out

engineered_features = [
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "CREDIT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY",
    "GOODS_TO_CREDIT",
    "INCOME_PER_FAMILY_MEMBER",
    "INCOME_PER_CHILD",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MAX"
]

X_train_fe[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,246008.0,43.886423,11.944884,20.503765,33.952088,43.104723,53.861739,6.907324e+01
EMPLOYED_YEARS,201865.0,6.529758,6.405822,-0.000000,2.097194,4.511978,8.695414,4.904038e+01
CREDIT_TO_INCOME,246008.0,3.959396,2.687597,0.004808,2.018667,3.268948,5.168514,4.922720e+01
ANNUITY_TO_INCOME,245998.0,0.180911,0.094571,0.000224,0.114583,0.162833,0.229000,1.570600e+00
CREDIT_TO_ANNUITY,245998.0,21.624291,7.820534,8.036674,15.647004,20.000000,27.099985,4.530508e+01
GOODS_TO_CREDIT,245787.0,0.900686,0.096646,0.166667,0.834725,0.893815,1.000000,6.666667e+00
INCOME_PER_FAMILY_MEMBER,246006.0,93113.103330,106834.975810,3375.000000,47250.000000,75000.000000,112500.000000,3.900000e+07
INCOME_PER_CHILD,246008.0,139533.126303,153274.718221,3600.000000,78750.000000,117000.000000,180000.000000,5.850000e+07
EXT_SOURCE_MEAN,245869.0,0.509119,0.149952,0.000011,0.413572,0.524241,0.622855,8.789034e-01
EXT_SOURCE_MAX,245869.0,0.615707,0.156299,0.000011,0.540016,0.648241,0.725276,9.516240e-01


In [9]:
#Check missing values created by feature engineering

engineered_missing = (
    X_train_fe[engineered_features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

engineered_missing

EMPLOYED_YEARS              44143
GOODS_TO_CREDIT               221
EXT_SOURCE_MEAN               139
EXT_SOURCE_MAX                139
ANNUITY_TO_INCOME              10
CREDIT_TO_ANNUITY              10
INCOME_PER_FAMILY_MEMBER        2
AGE_YEARS                       0
CREDIT_TO_INCOME                0
INCOME_PER_CHILD                0
dtype: int64

In [10]:
#Rebuild the preprocessing pipeline 

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features_fe = X_train_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_fe = X_train_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numeric_features_fe))
print("Categorical features:", len(categorical_features_fe))

Numerical features: 114
Categorical features: 16


In [11]:
#New preprocessing pipeline with engineered features

numeric_pipeline_fe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline_fe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor_fe = ColumnTransformer([
    ("num", numeric_pipeline_fe, numeric_features_fe),
    ("cat", categorical_pipeline_fe, categorical_features_fe)
])

In [12]:
preprocessor_fe.fit(X_train_fe)

print("Feature-engineering preprocessor fitted.")

Feature-engineering preprocessor fitted.


In [13]:
X_train_processed = preprocessor_fe.transform(X_train_fe)
X_valid_processed = preprocessor_fe.transform(X_valid_fe)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)

Processed training shape: (246008, 254)
Processed validation shape: (61503, 254)


In [14]:
#Save the updated preprocessor

import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    preprocessor_fe,
    MODEL_DIR / "baseline_preprocessor.joblib"
)

print("Updated baseline preprocessor saved.")

Updated baseline preprocessor saved.


In [15]:
#Save the engineered feature list

feature_metadata = {
    "original_feature_count": 120,
    "engineered_features": engineered_features,
    "total_features_before_encoding": X_train_fe.shape[1],
    "numeric_features": numeric_features_fe,
    "categorical_features": categorical_features_fe
}

feature_metadata

{'original_feature_count': 120,
 'engineered_features': ['AGE_YEARS',
  'EMPLOYED_YEARS',
  'CREDIT_TO_INCOME',
  'ANNUITY_TO_INCOME',
  'CREDIT_TO_ANNUITY',
  'GOODS_TO_CREDIT',
  'INCOME_PER_FAMILY_MEMBER',
  'INCOME_PER_CHILD',
  'EXT_SOURCE_MEAN',
  'EXT_SOURCE_MAX'],
 'total_features_before_encoding': 130,
 'numeric_features': ['CNT_CHILDREN',
  'AMT_INCOME_TOTAL',
  'AMT_CREDIT',
  'AMT_ANNUITY',
  'AMT_GOODS_PRICE',
  'REGION_POPULATION_RELATIVE',
  'DAYS_BIRTH',
  'DAYS_EMPLOYED',
  'DAYS_REGISTRATION',
  'DAYS_ID_PUBLISH',
  'OWN_CAR_AGE',
  'FLAG_MOBIL',
  'FLAG_EMP_PHONE',
  'FLAG_WORK_PHONE',
  'FLAG_CONT_MOBILE',
  'FLAG_PHONE',
  'FLAG_EMAIL',
  'CNT_FAM_MEMBERS',
  'REGION_RATING_CLIENT',
  'REGION_RATING_CLIENT_W_CITY',
  'HOUR_APPR_PROCESS_START',
  'REG_REGION_NOT_LIVE_REGION',
  'REG_REGION_NOT_WORK_REGION',
  'LIVE_REGION_NOT_WORK_REGION',
  'REG_CITY_NOT_LIVE_CITY',
  'REG_CITY_NOT_WORK_CITY',
  'LIVE_CITY_NOT_WORK_CITY',
  'EXT_SOURCE_1',
  'EXT_SOURCE_2',
  'EXT_

In [16]:
#save the feature metadata to a JSON file

import json

with open(
    "../models/baseline_feature_metadata.json",
    "w"
) as f:
    json.dump(feature_metadata, f, indent=4)

print("Feature metadata saved.")

Feature metadata saved.
